# **Construction of BioPAX-based Projected and Flatenned Pathway Networks**

This notebook also produces the AT_MN family of artifacts 

- BioPax: Biological Pathway Exchange

# Import Libraries and Configurations

In [1]:
import os
import sys
from collections import defaultdict
from itertools import count

import networkx as nx
import pandas as pd
from pybiopax import model_from_owl_file
from pybiopax.biopax import (
    Complex,
    Pathway,
    Protein,
)

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    DATA_SUBDIRS,
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
    REACTOME_BIOPAX_HSA_FILE,
    REACTOME_DATA_DIRS,
)

# Dictionary to index pathway objects by Reactome ID
PATHWAYS = dict()

# Counter for pathway locations ID creation
LOCATIONS_COUNTER = count(1)

# Dynamic programming caches
INTERACTION_CACHE = dict()
PROTEIN_COLLECTION_CACHE = dict()
LOCATIONS_CACHE = dict()

# Functions

## Pathway Network Projection

### Collect Proteins

In [2]:
def collect_proteins(entity):
    """
    Recursively collect protein objects from a BioPAX physical entity.

    This function traverses a BioPAX physical entity hierarchy to collect
    all protein objects contained directly or indirectly within a given
    entity. It supports individual proteins, protein entity sets, and
    protein complexes. The collected proteins are stored in a global
    cache to avoid redundant traversals of previously processed entities.

    Parameters
    ----------
    entity : Protein or Complex or None
        The BioPAX physical entity from which proteins will be collected.
        May be an individual protein, a protein entity set, a protein
        complex, or None.

    Returns
    -------
    set of Protein
        Set containing the unique protein objects collected from the
        physical entity. Returns an empty set if the entity is None
        or no proteins are found.

    Notes
    -----
    - The following global configuration variable is used:
        * `PROTEIN_COLLECTION_CACHE` : Dictionary storing previously
          collected protein objects for each physical entity.
    - Protein entity sets are recursively traversed through their
      `member_physical_entity` attribute.
    - Protein complexes are recursively traversed through their
      `component` attribute.
    - For individual protein objects, the protein itself is collected
      only when no member physical entities are defined.
    - Cache entries are indexed by the entity's `uid` attribute when
      available, or by its Python object ID otherwise.
    - Duplicate protein objects are automatically removed by using
      a set.

    Examples
    --------
    >>> proteins = collect_proteins(complex_entity)
    >>> len(proteins)
    5
    """
    # None object
    if entity is None:
        return set()

    # Get entity object ID
    entity_id = getattr(entity, 'uid', id(entity))

    # Check cache
    if entity_id in PROTEIN_COLLECTION_CACHE:
        return PROTEIN_COLLECTION_CACHE[entity_id]

    # Create a set to store Protein objects
    proteins = set()

    # Get EntitySet members, if present
    member_entities = getattr(
        entity, 'member_physical_entity', None
    )

    # Protein object
    if isinstance(entity, Protein):
        if member_entities:
            for member in member_entities:
                proteins.update(
                    collect_proteins(member)
                )
        else:
            proteins.add(entity)

    # Complex object
    if isinstance(entity, Complex):
        for component in entity.component:
            proteins.update(
                collect_proteins(component)
            )

    # Cache result
    PROTEIN_COLLECTION_CACHE[entity_id] = proteins

    return proteins

### Get Interaction Proteins

In [3]:
def get_interaction_proteins(interaction):
    """
    Collect protein objects participating in a BioPAX interaction.

    This function extracts all proteins associated with a given BioPAX
    interaction by traversing its controller, left, and right participants.
    Protein objects are recursively collected from physical entities using
    `collect_proteins()`, allowing proteins contained within entity sets
    and complexes to be included. The collected proteins are stored in a
    global cache to avoid redundant traversals of previously processed
    interactions.

    Parameters
    ----------
    interaction : object
        The BioPAX interaction from which proteins will be collected.
        Must expose one or more of the `controller`, `left`, and `right`
        attributes, each containing an iterable of physical entities.

    Returns
    -------
    set of Protein
        Set containing the unique protein objects collected from the
        interaction participants. Returns an empty set if no proteins
        are found.

    Notes
    -----
    - The following global configuration variable is used:
        * `INTERACTION_CACHE` : Dictionary storing previously collected
          protein objects for each interaction.
    - The function examines the `controller`, `left`, and `right`
      attributes of the interaction, skipping missing or empty attributes.
    - Protein objects are recursively collected using `collect_proteins()`,
      which supports individual proteins, protein entity sets, and
      protein complexes.
    - Cache entries are indexed by the interaction's `uid` attribute
      when available, or by its Python object ID otherwise.
    - Duplicate protein objects are automatically removed by using
      a set.

    Examples
    --------
    >>> proteins = get_interaction_proteins(interaction)
    >>> len(proteins)
    12
    """
    # Get interaction object ID
    interaction_id = getattr(
        interaction, 'uid', id(interaction)
    )

    # Check if the interaction has been visited previously
    if interaction_id in INTERACTION_CACHE:
        return INTERACTION_CACHE[interaction_id]

    # Create a set to store Protein objects
    proteins = set()

    # Iterate over the attributes of interest
    for attr in ('controller', 'left', 'right'):
        # Get the object(s) contained in the attribute
        entities = getattr(interaction, attr, None)

        # Attribute with an empty object
        if not entities:
            continue
        
        # Iterate over entity(ies) to extract its proteins
        for entity in entities:
            proteins.update(
                collect_proteins(entity)
            )

    # Add interaction and its proteins to the cache
    INTERACTION_CACHE[interaction_id] = proteins

    return proteins

### Build Projected Network

In [4]:
def pathway_network_projection(G, pathway, parent_id, visited=None):
    """
    Project a BioPAX pathway hierarchy into a directed network.

    This function recursively traverses a BioPAX pathway to construct a
    directed network representing its hierarchical organization and
    associated biological entities. The network integrates pathway steps,
    biological processes, nested pathways, and participating proteins.
    Nodes are annotated with their names and biological types, while
    edges represent hierarchical or participation relationships. The
    graph is modified in place and returned after traversal.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph in which the pathway hierarchy will be represented.
        The graph is modified in place.
    pathway : Pathway
        The BioPAX pathway to be traversed and projected into the graph.
    parent_id : str
        Identifier of the parent node to which the pathway steps will
        be connected.
    visited : set, optional
        Set containing the identifiers of previously visited pathways.
        If None, an empty set is initialized.

    Returns
    -------
    networkx.DiGraph
        Directed graph containing the projected pathway hierarchy,
        including pathway steps, biological processes, and proteins.

    Notes
    -----
    - The function traverses pathway components through the
      `pathway_order` attribute and their associated processes through
      the `step_process` attribute.
    - The network contains the following types of relationships:
        * Pathway to pathway step.
        * Pathway step to biological process.
        * Biological process to participating protein.
    - Nested pathways are recursively traversed and connected through
      their corresponding pathway process nodes.
    - Protein objects associated with biological processes are collected
      using `get_interaction_proteins()`.
    - Each node is annotated with `name` and `type` attributes.
      Control instances may also include a `controlled` attribute
      identifying the controlled biological process.
    - Existing nodes are reused based on their unique identifiers,
      allowing biological entities to participate in multiple relationships.
    - Previously visited pathways are skipped to prevent redundant
      traversal and infinite recursion.
    - Pathways without a `pathway_order` attribute or with an empty
      pathway order are not expanded.
    - The graph is modified in place. The function does not explicitly
      validate whether the resulting directed network is acyclic.

    Examples
    --------
    >>> G = nx.DiGraph()
    >>> G.add_node(root_pathway.uid, type='Pathway')
    >>> G = pathway_network_projection(
    ...     G,
    ...     pathway=root_pathway,
    ...     parent_id=root_pathway.uid
    ... )
    >>> G.number_of_nodes()
    153
    """
    # Check if the set of visited pathways has been initialized
    if visited is None:
        visited = set()

    # Get pathway object ID
    pathway_id = getattr(pathway, 'uid', id(pathway))

    # Check if the pathway has already been visited
    if pathway_id in visited:
        return G

    # Add the pathway to the set of visited pathways
    visited.add(pathway_id)

    # Get the order of the pathway components
    pathway_order = getattr(pathway, 'pathway_order', None)
    
    if not pathway_order:
        return G

    # Iterate over the pathway step(s)
    for step in pathway_order:
        # Add pathway step node to the graph, if necessary
        if step.uid not in G:
            G.add_node(
                step.uid,
                name=getattr(
                    step, 'display_name', step.uid
                ),
                type='PathwayStep'
            )

        # Add edge from pathway to pathway step
        G.add_edge(parent_id, step.uid)

        # Iterate over the process(es) in the pathway step
        for process in step.step_process:
            # Add process node to the graph, if necessary
            if process.uid not in G:
                process_name = (
                    getattr(process, 'display_name', None)
                    or process.uid
                )

                if process_name == process.uid:
                    process_name = (
                        getattr(process, 'control_type', None)
                        or process.uid
                    )

                node_attributes = {
                    'name': process_name,
                    'type': type(process).__name__
                }

                # Store controlled attribute for Control instances
                if hasattr(process, 'controlled'):
                    controlled = getattr(process, 'controlled', None)

                    if controlled is not None:
                        node_attributes['controlled'] = (
                            controlled.uid
                            if hasattr(controlled, 'uid')
                            else controlled
                        )

                G.add_node(
                    process.uid,
                    **node_attributes
                )

            # Add edge from pathway step to process
            G.add_edge(step.uid, process.uid)

            # Nested Pathway object
            if isinstance(process, Pathway):
                # Traverse the subpathway
                pathway_network_projection(
                    G=G,
                    pathway=process,
                    parent_id=process.uid,
                    visited=visited
                )
                continue

            # Get proteins related to the Interaction object
            proteins = get_interaction_proteins(process)

            # Iterate over the proteins
            for protein in proteins:
                # Add protein node to the graph, if necessary
                if protein.uid not in G:
                    protein_name = (
                        getattr(protein, 'display_name', None)
                        or protein.uid
                    )

                    G.add_node(
                        protein.uid,
                        name=protein_name,
                        type='Protein'
                    )

                # Add edge from process to protein
                G.add_edge(process.uid, protein.uid)

    return G

## Pathway Network Flattening

### Merge Control and Reaction Nodes

In [5]:
def merge_control_reaction_nodes(G):
    """
    Merge BioPAX Control nodes with their controlled reaction nodes.

    This function identifies Control nodes associated with biochemical
    reactions and merges them when both nodes share the same external
    predecessors. The merged nodes preserve the relevant attributes and
    relationships of the original nodes, reducing redundant representations
    of biological processes in the pathway network. The operation is
    performed on a copy of the input graph, preserving the original network.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph containing BioPAX Control and reaction nodes.
        Control nodes must include a `controlled` attribute identifying
        their associated reaction nodes.

    Returns
    -------
    networkx.DiGraph
        Modified copy of the directed graph in which eligible Control
        and reaction nodes are replaced by merged nodes.

    Notes
    -----
    - Control nodes are identified by the presence of a `controlled`
      attribute referencing another node in the graph.
    - Two nodes are merged only when their external predecessor sets
      are identical, excluding direct relationships between them.
    - Each merged node has the following attributes:
        * `name` : Combined names of the reaction and Control nodes.
        * `type` : Combined biological types of the original nodes.
        * `control_type` : Control type of the original Control node,
          when available.
    - Merged node identifiers follow the format
      `merge_<control_id>_<reaction_id>`.
    - Incoming and outgoing edges of both original nodes are inherited
      by the merged node, excluding relationships internal to the pair.
    - Duplicate edges are avoided, and edge attributes are preserved
      from the first occurrence of each inherited relationship.
    - Original Control and reaction nodes are removed after merging.
    - The original graph remains unchanged.

    Examples
    --------
    >>> merged_G = merge_control_reaction_nodes(G)
    >>> merged_G.number_of_nodes()
    120
    """
    # Preserve the original graph
    G = G.copy()

    # Identify Control nodes with a controlled attribute
    control_nodes = [
        node_id
        for node_id, data in G.nodes(data=True)
        if 'controlled' in data
    ]

    # Iterate over the Control nodes
    for control_id in control_nodes:
        # Skip Control nodes previously removed
        if control_id not in G:
            continue

        # Get the controlled reaction ID
        control_data = G.nodes[control_id]
        reaction_id = control_data['controlled']

        # Skip reactions not represented in the graph
        if reaction_id not in G:
            continue

        # Get the controlled reaction attributes
        reaction_data = G.nodes[reaction_id]

        # Get the predecessors of both nodes
        control_preds = set(G.predecessors(control_id))
        reaction_preds = set(G.predecessors(reaction_id))

        # Ignore direct relationships between Control and reaction
        control_preds.discard(reaction_id)
        reaction_preds.discard(control_id)

        # Check whether both nodes share the same external predecessors
        if control_preds != reaction_preds:
            continue

        # Create the merged node ID
        merged_id = f'merge_{control_id}_{reaction_id}'

        # Define the merged node name
        merged_name = (
            f"{reaction_data.get('name', reaction_id)} "
            f"[{control_data.get('name', control_id)}]"
        )

        # Define the merged node type
        control_type = control_data.get('type', 'Control')
        reaction_type = reaction_data.get('type', 'Reaction')
        merged_type = f'{control_type}_{reaction_type}'

        # Add the merged node to the graph
        G.add_node(
            merged_id,
            name=merged_name,
            type=merged_type,
            control_type=control_data.get('control_type'),
        )

        # Create a dictionary to store inherited edges
        edges = {}

        # Define a helper function to avoid duplicate edges
        def add_edge(u, v, attrs):
            if (u, v) not in edges:
                edges[(u, v)] = dict(attrs)

        # Collect incoming edges from the Control node
        for source, _, attrs in G.in_edges(control_id, data=True):
            if source not in (control_id, reaction_id):
                add_edge(source, merged_id, attrs)

        # Collect outgoing edges from the Control node
        for _, target, attrs in G.out_edges(control_id, data=True):
            if target not in (control_id, reaction_id):
                add_edge(merged_id, target, attrs)

        # Collect incoming edges from the reaction node
        for source, _, attrs in G.in_edges(reaction_id, data=True):
            if source not in (control_id, reaction_id):
                add_edge(source, merged_id, attrs)

        # Collect outgoing edges from the reaction node
        for _, target, attrs in G.out_edges(reaction_id, data=True):
            if target not in (control_id, reaction_id):
                add_edge(merged_id, target, attrs)

        # Remove the original Control and reaction nodes
        G.remove_nodes_from([control_id, reaction_id])

        # Add the inherited edges to the graph
        for (u, v), attrs in edges.items():
            if not G.has_edge(u, v):
                G.add_edge(u, v, **attrs)

    return G

### Project Pathway Steps

In [6]:
def project_pathway_steps(G):
    """
    Project a pathway network by removing intermediate pathway step nodes.

    This function simplifies a directed pathway network by removing all
    PathwayStep nodes while preserving connectivity between their
    predecessors and successors. For each pathway step, new directed edges
    are created between its parent and child nodes before the step is
    removed. The operation is performed on a copy of the input graph,
    preserving the original network.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph representing a pathway network, with nodes
        annotated by a `type` attribute identifying their biological
        or structural type.

    Returns
    -------
    networkx.DiGraph
        Modified copy of the directed graph in which PathwayStep nodes
        are removed and their predecessors and successors are directly
        connected.

    Notes
    -----
    - PathwayStep nodes are identified by the `type` attribute with
      the value `PathwayStep`.
    - For each PathwayStep node, directed edges are created between
      every predecessor and every successor.
    - Existing edges between predecessor-successor pairs are reused,
      avoiding duplicate edges.
    - Newly created edges do not inherit attributes from the edges
      connected to the removed PathwayStep nodes.
    - The projection preserves directed reachability between remaining
      nodes but does not explicitly enforce acyclicity.
    - The original graph remains unchanged.

    Examples
    --------
    >>> G_projected = project_pathway_steps(G)
    >>> G_projected.number_of_nodes()
    120

    >>> any(
    ...     attr['type'] == 'PathwayStep'
    ...     for _, attr in G_projected.nodes(data=True)
    ... )
    False
    """
    # Create a copy of the graph
    G_proj = G.copy()

    # Get the pathway step nodes
    step_nodes = [
        n for n, attr in G_proj.nodes(data=True)
        if attr['type'] == 'PathwayStep'
    ]

    # Iterate over each pathway step node
    for step in step_nodes:
        # Get node parent(s) and child(ren)
        parents = tuple(G_proj.predecessors(step))
        children = tuple(G_proj.successors(step))

        # Add edges between parent(s) and child(ren)
        G_proj.add_edges_from(
            (p, c)
            for p in parents
            for c in children
        )

        # Remove the node from the graph
        G_proj.remove_node(step)

    return G_proj

### Identify Protein-Pathway Paths

In [7]:
def identify_pp_paths(G, source):
    """
    Enumerate all pathway-to-protein (PP) paths from a source node.

    This function traverses a directed acyclic graph (DAG) in topological
    order and yields every directed path from the specified source node to
    each reachable protein node. Protein nodes are identified by their
    ``type`` attribute equal to ``"Protein"``.

    Paths are propagated incrementally through the graph, allowing every
    distinct route from the source to a protein to be enumerated without
    performing repeated graph traversals.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed acyclic graph containing nodes annotated with a
        ``type`` attribute.
    source : hashable
        Identifier of the source node from which pathway-to-protein paths are
        generated.

    Yields
    ------
    list
        A list of node identifiers representing a directed pathway-to-protein
        (PP) path from ``source`` to a protein node. Each yielded path begins
        with ``source`` and ends with a node whose ``type`` is
        ``"Protein"``.

    Notes
    -----
    This function assumes that ``G`` is a directed acyclic graph (DAG). The
    traversal relies on :func:`networkx.topological_sort` and propagates
    partial paths through successor nodes until protein leaves are reached.

    Examples
    --------
    >>> for path in identify_pp_paths(G, source='R-HSA-12345'):
    ...     print(path)
    ['R-HSA-12345', 'reaction1', 'P12345']
    ['R-HSA-12345', 'reaction2', 'complex1', 'P67890']
    """
    # Cache useful objects
    type = nx.get_node_attributes(G, 'type')
    successors = G.successors

    # Create a dictionary of empty lists to store paths
    paths = defaultdict(list)

    # Initialize the list of paths from the source node
    paths[source].append([source])

    # Iterate over the nodes in topological order
    for node in nx.topological_sort(G):
        # Get all paths that lead to the node
        current_paths = paths.get(node)

        # Skip unreachable node
        if not current_paths:
            continue

        # Output protein (leaf) paths
        if type[node] == 'Protein':
            yield from current_paths
            continue

        # Iterate over the node's child(ren)
        for child in successors(node):
            # Cache the list of paths from the child node
            child_paths = paths[child]

            # Iterate over the paths the lead to the node
            for path in current_paths:
                # Propagate paths to child
                child_paths.append(path + [child])

### Build Flattened Network

In [8]:
def pathway_network_flattening(G, pathway_id):
    """
    Flatten a pathway network into a pathway-location-mRNA representation.

    This function transforms a directed pathway network into a simplified
    tripartite representation connecting a biological pathway to its
    associated messenger RNAs through semantic pathway locations. The
    network is constructed by merging Control and reaction nodes, removing
    intermediate pathway steps, and identifying all pathway-to-protein
    paths. Each path is converted into a textual representation of its
    intermediate biological processes, defining a pathway location.
    The resulting network is represented as a directed graph.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph representing the hierarchical organization of a
        biological pathway, including pathway steps, biological processes,
        and participating proteins.
    pathway_id : str
        Identifier of the pathway to be flattened. Must be a key in
        the global `PATHWAYS` dictionary.

    Returns
    -------
    networkx.DiGraph
        Directed graph representing the flattened tripartite network,
        containing three types of nodes:
        - `Pathway` : Root biological pathway.
        - `Pathway Location` : Semantic representation of intermediate
          biological processes along a pathway-to-protein path.
        - `Messenger RNA` : Messenger RNA representation of a protein
          identified in the pathway network.

    Notes
    -----
    - The following global variables are used:
        * `PATHWAYS` : Dictionary indexing BioPAX pathway objects by
          their identifiers.
        * `LOCATIONS_CACHE` : Dictionary mapping semantic pathway
          locations to their unique identifiers.
        * `LOCATIONS_COUNTER` : Counter used to generate unique
          pathway location identifiers.
    - Control and reaction nodes are merged using
      `merge_control_reaction_nodes()`.
    - Intermediate PathwayStep nodes are removed using
      `project_pathway_steps()`.
    - Pathway-to-protein paths are identified using
      `identify_pp_paths()`, starting from the specified pathway.
    - Pathway locations are defined by concatenating the names of
      intermediate biological processes using ` > ` as a separator.
    - Identical semantic pathway locations share the same node
      identifier, including across function calls that reuse the
      global location cache.
    - Pathway location identifiers follow the format `PL00001`,
      `PL00002`, and so on.
    - Protein nodes are represented as `Messenger RNA` nodes using
      their names as both identifiers and labels.
    - The flattened network contains directed relationships from
      pathways to pathway locations and from pathway locations to
      messenger RNAs.
    - The input graph remains unchanged.

    Examples
    --------
    >>> T = pathway_network_flattening(G, 'R-HSA-123456')
    >>> T.number_of_nodes()
    45

    >>> T.number_of_edges()
    62

    >>> from collections import Counter
    >>> Counter(nx.get_node_attributes(T, 'type').values())
    Counter({
        'Messenger RNA': 25,
        'Pathway Location': 19,
        'Pathway': 1
    })
    """
    pathway_obj = PATHWAYS[pathway_id]

    # Merge control and reaction nodes
    G_merged = merge_control_reaction_nodes(G)

    # Project the pathway step nodes
    G_proj = project_pathway_steps(G_merged)

    # Create an object to represent the flattened network
    T = nx.DiGraph()

    # Add the pathway node to the flattened network
    nodes = G_proj.nodes
    pathway_name = nodes[pathway_obj.uid]['name']
    T.add_node(
        pathway_id,
        label=pathway_name,
        type='Pathway'
    )

    # Iterate over each pathway-protein (PP) path
    for path in identify_pp_paths(
        G=G_proj, source=pathway_obj.uid,
    ):
        # Get protein name
        protein = nodes[path[-1]]['name']

        # Add protein node to the network, if necessary
        if protein not in T:
            T.add_node(
                protein,
                label=protein,
                type='Messenger RNA'
            )
        
        # Create PP path textual representation (pathway location)
        location = ' > '.join(
            nodes[n]['name']
            for n in path[1:-1]
        )

        # Retrieve or create pathwat location ID
        if location not in LOCATIONS_CACHE:
            # Create pathway location ID
            location_id = (
                f'PL{next(LOCATIONS_COUNTER):05d}'
            )

            # Cache pathway location ID
            LOCATIONS_CACHE[location] = location_id
        else:
            # Retrieve the pathway location ID
            location_id = LOCATIONS_CACHE[location]

        # Add location node to the network, if necessary
        if location_id not in T:
            T.add_node(
                location_id,
                label=location,
                type='Pathway Location'
            )

        # Add edge from pathway to pathway location
        T.add_edge(pathway_id, location_id)

        # Add edge from pathway location to protein
        T.add_edge(location_id, protein)

    return T

## Controller

### Export Network Tables

In [9]:
def export_network_tables(G, pathway_id, network_type):
    """
    Export a pathway network as edge and node tables.

    This function converts a directed pathway network into two DataFrames
    containing its edges and nodes with their associated attributes. The
    resulting tables are stored in processed CSV files according to the
    pathway identifier and network type. The file paths and filenames
    are obtained from global configuration dictionaries.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph representing the pathway network to be exported.
    pathway_id : str
        Identifier of the biological pathway associated with the network.
        Used as a prefix for the output filenames.
    network_type : str
        Name of the subdirectory corresponding to the pathway network
        type within `REACTOME_DATA_DIRS['processed']`.

    Returns
    -------
    edges_df : pandas.DataFrame
        DataFrame containing the directed edges of the pathway network,
        with the following columns:
        - `source` : Identifier of the source node.
        - `target` : Identifier of the target node.
        Additional columns represent edge attributes, when available.

    nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the pathway network, with
        an `id` column identifying each node and additional columns
        representing its associated attributes.

    Notes
    -----
    - The following global configuration variables are used:
        * `NETWORK_FILES` : Filenames for specific network data files.
        * `REACTOME_DATA_DIRS` : Directory paths for processed Reactome
          pathway network data.
    - The edge table is generated using `networkx.to_pandas_edgelist()`,
      preserving edge attributes when available.
    - The node table is constructed from the graph nodes and their
      associated attributes.
    - The output directory is constructed by joining the processed
      Reactome data directory with the specified network type.
    - Output filenames are generated by concatenating the pathway
      identifier with the corresponding filename configuration.
    - The function overwrites the corresponding edge and node CSV files
      in the processed directory for the specified network type.
    - The input graph remains unchanged.

    Examples
    --------
    >>> edges_df, nodes_df = export_network_tables(
    ...     G,
    ...     pathway_id='R-HSA-123456',
    ...     network_type='flattened'
    ... )
    >>> edges_df[['source', 'target']].head()
    >>> nodes_df[['id', 'label', 'type']].head()
    """
    # Create a DataFrame for the edge list
    edges_df = nx.to_pandas_edgelist(G)

    # Create a DataFrame for the node attributes
    nodes_df = pd.DataFrame(
        (
            {'id': n, **attrs}
            for n, attrs in G.nodes(data=True)
        )
    )

    # Define the directory path
    dir_path = os.path.join(
        REACTOME_DATA_DIRS['processed'], network_type
    )
    os.makedirs(dir_path, exist_ok=True)

    # Store the DataFrame of edges
    file_name = pathway_id + NETWORK_FILES['pathway-edges']
    edges_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    # Store the DataFrame of nodes
    file_name = pathway_id + NETWORK_FILES['pathway-nodes']
    nodes_df.to_csv(
        os.path.join(dir_path, file_name), index=False
    )

    return edges_df, nodes_df

### Build Pathway Network

In [10]:
def build_pathway_network(pathway_id):
    """
    Build projected and flattened networks for a biological pathway.

    This function retrieves a BioPAX pathway from a global collection and
    constructs two complementary network representations. The projected
    network preserves the hierarchical relationships between pathway steps,
    biological processes, and participating proteins. The flattened network
    simplifies this representation by connecting the pathway to messenger
    RNAs through semantic pathway locations. Both networks are exported
    as edge and node tables to processed CSV files.

    Parameters
    ----------
    pathway_id : str
        Identifier of the biological pathway for which the networks will
        be constructed. Must be a key in the global `PATHWAYS` dictionary.

    Returns
    -------
    flattened_edges_df : pandas.DataFrame
        DataFrame containing the directed edges of the flattened pathway
        network, representing pathway-location and location-mRNA
        relationships.

    flattened_nodes_df : pandas.DataFrame
        DataFrame containing the nodes of the flattened pathway network,
        including pathways, pathway locations, and messenger RNAs, with
        their associated attributes.

    Raises
    ------
    KeyError
        If the specified pathway identifier is not present in the
        global `PATHWAYS` dictionary.

    Notes
    -----
    - The following global variable is used:
        * `PATHWAYS` : Dictionary indexing BioPAX pathway objects by
          their identifiers.
    - The projected pathway network is constructed using
      `pathway_network_projection()`, starting from the root pathway.
    - The flattened pathway network is constructed using
      `pathway_network_flattening()`, which merges Control and reaction
      nodes, removes intermediate PathwayStep nodes, and represents
      pathway-to-protein paths as semantic pathway locations.
    - Both networks are exported using `export_network_tables()`.
    - Projected network files are stored in the `projected-networks`
      subdirectory, while flattened network files are stored in the
      `flattened-networks` subdirectory.
    - The function returns only the edge and node DataFrames of the
      flattened network.
    - Existing CSV files for the specified pathway are overwritten.

    Examples
    --------
    >>> edges_df, nodes_df = build_pathway_network('R-HSA-123456')
    >>> edges_df[['source', 'target']].head()
    >>> nodes_df[['id', 'label', 'type']].head()
    """
    # Check whether the pathway is indexed
    if pathway_id not in PATHWAYS:
        raise KeyError(
            f'Pathway ID {pathway_id!r} not found in PATHWAYS.'
        )

    # Get the pathway object
    pathway_obj = PATHWAYS[pathway_id]

    # Create a directed graph to represent the pathway hierarchy
    G = nx.DiGraph()

    # Add the root pathway node to the graph
    G.add_node(
        pathway_obj.uid,
        name=pathway_obj.display_name,
        type='Pathway',
    )

    # Build the projected pathway network
    G = pathway_network_projection(
        G=G,
        pathway=pathway_obj,
        parent_id=pathway_obj.uid,
    )

    # Export the projected pathway network
    projected_edges_df, projected_nodes_df = \
        export_network_tables(G, pathway_id, 'projected-networks')

    # Build the flattened pathway network
    T = pathway_network_flattening(G, pathway_id)

    # Export the flattened pathway network
    flattened_edges_df, flattened_nodes_df = \
        export_network_tables(T, pathway_id, 'flattened-networks')

    return flattened_edges_df, flattened_nodes_df

# Create the BioPAX Model

In [11]:
# Create a BioPAX model from the OWL file
file_name = REACTOME_BIOPAX_HSA_FILE
dir_path = REACTOME_DATA_DIRS['external']
model = model_from_owl_file(os.path.join(dir_path, file_name))

Processing OWL elements:   0%|          | 0.00/545k [00:00<?, ?it/s]

# Index all *Homo sapiens* Pathways

In [12]:
# Iterate over the Pathway objects
for pathway in model.get_objects_by_type(Pathway):
    # Iterate over the xrefs of the Pathway object
    for xref in pathway.xref:
        # Add the Reactome Pathway object to the dictionary
        if getattr(xref, 'db', None) == 'Reactome':
            reactome_id = xref.id
            PATHWAYS[reactome_id] = pathway
            break

# Iterate Over the Pathways of Interest

In [13]:
# Create a list to store the pathways of interest
pathways = list()

# Define the nodes file name and the base directory path
file_name = NETWORK_FILES['tripartite-nodes']
dir_base_path = NETWORK_DATA_DIRS['processed']

# Iterate over the group node files
for group_dir in DATA_SUBDIRS:
    # Create a DataFrame for the nodes of the group's tripartite network
    dir_path = dir_base_path[group_dir]
    nodes_df = pd.read_csv(os.path.join(dir_path, file_name))

    # Extract the enriched pathways for the group
    group_pathways = (
        nodes_df
        .query('type == "Pathway"')
        ['id']
        .to_list()
    )

    # Add the enriched pathways for the group to the list
    pathways += group_pathways

# Remove duplicated pathways
pathways = set(pathways)

# Iterate over the pathways of interest
for pathway in pathways:
    # Remove the `REAC:` prefix
    pathway_id = pathway.replace('REAC:', '')

    # Build the pathway networks
    edges_df, nodes_df = build_pathway_network(pathway_id)

## Exploration of the Results

In [14]:
# Print the edges DataFrame of the last pathway
edges_df

,source,target
0,R-HSA-6804757,PL19101
1,R-HSA-6804757,PL19102
2,R-HSA-6804757,PL05317
3,R-HSA-6804757,PL19103
4,R-HSA-6804757,PL19104
...,...,...
204,PL19123,p-MDM2
205,PL19124,"p-S1981,Ac-K3016-ATM"
206,PL19124,p-MDM2
207,PL19124,MDM4


In [15]:
# Print the nodes DataFrame of the last pathway
nodes_df

,id,label,type
0,R-HSA-6804757,Regulation of TP53 Degradation,Pathway
1,"p-S,3T-CHEK2","p-S,3T-CHEK2",Messenger RNA
2,PL19101,Phosphorylation of MDM4 by CHEK2 [ACTIVATION],Pathway Location
3,RNF34,RNF34,Messenger RNA
4,PL19102,RNF34 or RFFL ubiquitinates phosphorylated TP5...,Pathway Location
...,...,...,...
81,"p-S1981,Ac-K3016-ATM","p-S1981,Ac-K3016-ATM",Messenger RNA
82,PL19124,MDM2 forms homo- or heterodimers [INHIBITION],Pathway Location
83,MDM4,MDM4,Messenger RNA
84,PolyUb-p53,PolyUb-p53,Messenger RNA


In [16]:
# Print the pathway locations related to the last pathway
(
    nodes_df
    .query('type == "Pathway Location"')
    ['label']
    .to_list()
)

['Phosphorylation of MDM4 by CHEK2 [ACTIVATION]',
 'RNF34 or RFFL ubiquitinates phosphorylated TP53 [ACTIVATION]',
 'mTORC2 phosphorylates SGK1 [ACTIVATION]',
 'Phosphorylation of MDM4 by ATM [ACTIVATION]',
 'USP2 deubiquitinates MDM2,MDM4 [ACTIVATION]',
 'USP2 binds the heterodimer of MDM2 and MDM4',
 'ATM phosphorylates MDM2 [ACTIVATION]',
 'DAXX binds Ub-MDM2 and USP7',
 'USP7 deubiquitinates MDM2 [ACTIVATION]',
 'USP7 deubiquitinates TP53 and counteracts MDM2 [ACTIVATION]',
 'TP53 binds the MDM2 gene',
 'TP53 stimulates MDM2 transcription [ACTIVATION]',
 'PP2A-PP2R5C dephosphorylates MDM2 [ACTIVATION]',
 'MDM2 translocates to the cytosol',
 'AKT phosphorylates MDM2 [ACTIVATION]',
 'SGK1 phosphorylates MDM2 [ACTIVATION]',
 'PDPK1 phosphorylates SGK1 [ACTIVATION]',
 'p14ARF forms a ternary complex with MDM2 and TP53',
 'INHIBITION',
 'p14ARF sequesters MDM2',
 'MDM2 binds TP53 [INHIBITION]',
 'TP53 binds the CCNG1 gene',
 'MDM2 ubiquitinates TP53 [ACTIVATION]',
 'TP53 stimulates CCNG